### GPUを確認する

In [ ]:
!nvidia-smi

### ComfyUIを新規インストールする

In [ ]:
%cd /content
!git clone https://github.com/Comfy-Org/ComfyUI.git
%cd /content/ComfyUI
!pip install -q -r requirements.txt

### インストール状況を確認する

In [ ]:
import os
import torch

print("ComfyUI:", os.path.isdir("/content/ComfyUI"))
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU使用可能:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"VRAM: {vram:.1f} GB")

### Pinggyをインストールする

In [ ]:
!pip install -q pinggy
print("Pinggyのインストールが完了しました")

### Hunyuan3D 2.1のモデルをダウンロードする

In [ ]:
!mkdir -p /content/ComfyUI/models/checkpoints

!wget -c \
  --show-progress \
  "https://huggingface.co/Comfy-Org/hunyuan3D_2.1_repackaged/resolve/main/hunyuan_3d_v2.1.safetensors" \
  -O "/content/ComfyUI/models/checkpoints/hunyuan_3d_v2.1.safetensors"

### ダウンロードファイルを確認します。

In [ ]:
from pathlib import Path

model_path = Path(
    "/content/ComfyUI/models/checkpoints/hunyuan_3d_v2.1.safetensors"
)

print("存在:", model_path.exists())

if model_path.exists():
    print(f"容量: {model_path.stat().st_size / 1024**3:.2f} GB")

### ComfyUIをバックグラウンドで起動する

In [ ]:
import os
import signal
import subprocess
import time
import requests

# 以前起動したComfyUIがあれば停止する
subprocess.run(
    ["pkill", "-f", "/content/ComfyUI/main.py"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(2)

# ログを保存してComfyUIをバックグラウンド起動する
log_file = open("/content/comfyui.log", "w")

comfyui_process = subprocess.Popen(
    [
        "python",
        "/content/ComfyUI/main.py",
        "--listen", "0.0.0.0",
        "--port", "8188",
        "--lowvram"
    ],
    cwd="/content/ComfyUI",
    stdout=log_file,
    stderr=subprocess.STDOUT
)

print("ComfyUIを起動しています……")

for _ in range(60):
    time.sleep(2)

    try:
        response = requests.get(
            "http://127.0.0.1:8188",
            timeout=3
        )

        if response.status_code == 200:
            print("ComfyUIの起動に成功しました")
            break

    except requests.RequestException:
        pass
else:
    print("起動を確認できませんでした。ログを確認してください")

### ブラウザで開くためのURLを発行する

In [ ]:
import pinggy

pinggy_tunnel = pinggy.start_tunnel(
    forwardto="localhost:8188"
)

print("\nComfyUIを開くURL：")

for url in pinggy_tunnel.urls:
    print(url)